In [1]:
import pandas as pd
import os

os.makedirs('../data/analytics', exist_ok=True)


In [2]:
suppliers = pd.read_csv(
    '../data/processed/suppliers.csv'
)

procurement = pd.read_csv(
    '../data/curated/procurement_curated.csv'
)

suppliers.head()

,supplier_id,supplier_name,supplier_type,product_category,city,province,region,pincode,lead_time_days,reliability_score,import_duty_rate,china_tax_id
0,SUP0001,SHENZHEN OEM SUPPLIES,OEM,MEDIUM PARTS,SHENZHEN,GUANGDONG,SOUTH CHINA,586808,19,5,8,90SUZHOU398241
1,SUP0002,GUANGZHOU LOCAL VENDOR SUPPLIES,LOCAL VENDOR,SMALL PARTS,GUANGZHOU,GUANGDONG,SOUTH CHINA,966203,10,5,12,89TIANJI357980
2,SUP0003,SHANGHAI DISTRIBUTOR SUPPLIES,DISTRIBUTOR,LARGE PARTS,SHANGHAI,SHANGHAI,EAST CHINA,655757,16,4,8,96SHENZH983302
3,SUP0004,NINGBO DISTRIBUTOR SUPPLIES,DISTRIBUTOR,LARGE PARTS,NINGBO,ZHEJIANG,EAST CHINA,536477,18,3,8,39GUANGZ946640
4,SUP0005,TIANJIN OEM SUPPLIES,OEM,SMALL PARTS,TIANJIN,TIANJIN,NORTH CHINA,334261,28,3,5,59GUANGZ986793


In [5]:
# Measure procurement spend by supplier
                     
supplier_contribution = (
    procurement.groupby('supplier_id')
    ['line_grand_total']
    .sum()
    .reset_index()
    .rename(
        columns={
            'line_grand_total':
            'total_procurement_spend'
        }
    )
    .sort_values(
        'total_procurement_spend',
        ascending=False
    )
)

supplier_contribution.head()



supplier_contribution.to_csv(
    '../data/analytics/supplier_contribution.csv',
    index=False
)

In [6]:
#Supplier Reliability Analysis
supplier_reliability = (
    suppliers[
        [
            'supplier_id',
            'supplier_name',
            'lead_time_days',
            'reliability_score'
        ]
    ]
    .sort_values(
        'reliability_score',
        ascending=False
    )
)

supplier_reliability.head()


supplier_reliability.to_csv(
    '../data/analytics/supplier_reliability.csv',
    index=False
)

In [7]:
# Supplier Performance Comparison

supplier_performance = (
    suppliers.copy()
)

supplier_performance[
    'performance_score'
] = (
    supplier_performance['reliability_score']
    /
    supplier_performance['lead_time_days']
)

supplier_performance = (
    supplier_performance.sort_values(
        'performance_score',
        ascending=False
    )
)

supplier_performance[
    [
        'supplier_name',
        'performance_score'
    ]
].head()

supplier_performance.to_csv(
    '../data/analytics/supplier_performance.csv',
    index=False
)

In [9]:
# Supplier Dependency Risk Analysis

dependency = (
    supplier_contribution.copy()
)

dependency[
    'spend_pct'
] = (
    dependency['total_procurement_spend']
    /
    dependency[
        'total_procurement_spend'
    ].sum()
) * 100

dependency.head()

dependency.to_csv(
    '../data/analytics/supplier_dependency_risk.csv',
    index=False
)

In [10]:
# Identify critical suppliers based on dependency and reliability


critical_suppliers = (
    dependency.merge(
        suppliers,
        on='supplier_id',
        how='left'
    )
)

critical_suppliers = (
    critical_suppliers[
        (
            critical_suppliers[
                'spend_pct'
            ] > 10
        )
        &
        (
            critical_suppliers[
                'reliability_score'
            ] < 8
        )
    ]
)

critical_suppliers.head()

critical_suppliers.to_csv(
    '../data/analytics/critical_suppliers.csv',
    index=False
)


In [13]:
# Supplier Diversification Analysis

supplier_diversification = (
    suppliers.groupby(
        'product_category'
    )
    .size()
    .reset_index(
        name='supplier_count'
    )
)

supplier_diversification

supplier_diversification.to_csv(
    '../data/analytics/supplier_diversification.csv',
    index=False
)